# Toxic Comment Classification

Using **BERT + Bi-GRU** pipeline for classifying toxic comments into three categories:
- **Insult**
- **Threat**
- **Sexual Harassment**

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import re

---
## Normalization (`clean_toxic_text`)

**Steps:**
1. Convert to lowercase.
2. Replace common leet characters (`3→e`, `1→i`, `0→o`, etc.).
3. Collapse repeated punctuation (e.g., `!!!` → `!`) to reduce noise.

In [ ]:
def clean_toxic_text(text):
    text = str(text).lower()
    replacements = {
        '3': 'e', '1': 'i', '0': 'o', '4': 'a', '5': 's', 
        '7': 't', '$': 's', '@': 'a', '8': 'b', 'x': 'x'
    }
    for char, rep in replacements.items():
        text = text.replace(char, rep)
    text = re.sub(r'([!?.])\ 1+', r'\1', text) 
    return text.strip()

# Example
print(clean_toxic_text("y0u 4r3 $tupid!!!"))  # → "you are stupid!"

---
## Dataset Class 

In [ ]:
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        # Returns total number of samples — used by DataLoader
        return len(self.texts)

    def __getitem__(self, item):
        # Ensure text is a string to avoid tokenizer errors
        text = str(self.texts[item])
        
        # Tokenize: converts text → input_ids + attention_mask
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,   # Adds [CLS] and [SEP] tokens
            max_length=self.max_len,
            padding='max_length',      # Pad shorter sequences
            truncation=True,           # Cut off sequences that are too long
            return_tensors='pt',
        )
        
        return {
            'ids': encoding['input_ids'].flatten(),
            'mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

---
## Model Architecture 

In [ ]:
class ToxicClassifier(nn.Module):
    def __init__(self, n_classes):
        super(ToxicClassifier, self).__init__()
        # Pre-trained BERT as the base encoder
        self.bert = AutoModel.from_pretrained('bert-base-uncased')
        # Bidirectional GRU: input=768 (BERT hidden size), hidden=256, output=512
        self.gru = nn.GRU(768, 256, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        # Output layer maps to 3 classes
        self.out = nn.Linear(512, n_classes)

    def forward(self, ids, mask):
        # Step 1: Get BERT's full sequence output (not just [CLS])
        outputs = self.bert(input_ids=ids, attention_mask=mask)
        last_hidden_state = outputs.last_hidden_state  # shape: [batch, seq_len, 768]
        
        # Step 2: Feed into Bi-GRU
        gru_out, _ = self.gru(last_hidden_state)       # shape: [batch, seq_len, 512]
        
        # Step 3: Max pooling — take the maximum value across the sequence dimension
        pooled_output, _ = torch.max(gru_out, dim=1)   # shape: [batch, 512]
        
        # Step 4: Dropout + classification head
        output = self.dropout(pooled_output)
        return self.out(output)                         # shape: [batch, 3]

---
## Train Model

1. Train/Val Split
- 85% training / 15% validation with `random_state=42` for reproducibility.


2. Optimizer & Loss**
- **AdamW** with `lr=2e-5`: The standard fine-tuning learning rate for BERT. Too high risks destroying pre-trained weights; too low converges too slowly.
- **`weight_decay=1e-6`**: Light L2 regularization.
- **CrossEntropyLoss**: Correct loss for multi-class classification.

In [ ]:
def train_model():
    # ── 4a. Load & preprocess ──────────────────────────────────────────
    df = pd.read_csv('/kaggle/input/competitions/dl-assignment-toxic-comment-classification/train.csv')
    df['Toxic Comment'] = df['Toxic Comment'].apply(clean_toxic_text)
    
    label_map = {"Insult": 0, "Threat": 1, "Sexual Harassment": 2}
    inv_label_map = {v: k for k, v in label_map.items()}  # {0: 'Insult', 1: 'Threat', ...}
    df['label'] = df['Classification'].map(label_map)

    # ── 4b. Train/Val split ────────────────────────────────────────────
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['Toxic Comment'].values, df['label'].values, test_size=0.15, random_state=42
    )

    # ── 4c. DataLoaders ───────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    train_loader = DataLoader(ToxicDataset(train_texts, train_labels, tokenizer, 80), batch_size=16, shuffle=True)
    val_loader   = DataLoader(ToxicDataset(val_texts, val_labels, tokenizer, 80),   batch_size=16)

    # ── 4d. Model, optimizer, loss ────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ToxicClassifier(n_classes=3).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=1e-6)
    loss_fn = nn.CrossEntropyLoss().to(device)

    print(f"Training on: {device}")

    # ── 4e. Training loop ─────────────────────────────────────────────
    for epoch in range(3):
        model.train()
        total_loss = 0
        for batch in train_loader:
            ids    = batch['ids'].to(device)
            mask   = batch['mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()           # Clear gradients from last step
            outputs = model(ids, mask)      # Forward pass
            loss = loss_fn(outputs, labels) # Compute loss
            loss.backward()                 # Backpropagate
            optimizer.step()                # Update weights
            total_loss += loss.item()
        
        # ── Validation ──────────────────────────────────────────────
        model.eval()  # Disables dropout during evaluation
        preds_list, labels_list = [], []
        with torch.no_grad():  # No gradients needed for inference
            for batch in val_loader:
                ids  = batch['ids'].to(device)
                mask = batch['mask'].to(device)
                outputs = model(ids, mask)
                _, preds = torch.max(outputs, dim=1)  # Argmax → predicted class
                preds_list.extend(preds.cpu().numpy())
                labels_list.extend(batch['labels'].numpy())
        
        val_acc = accuracy_score(labels_list, preds_list)
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")

    return model, tokenizer, device, inv_label_map

---
## Main

This is the entry point of the script. It:

1. **Trains** the model by calling `train_model()`.
3. **Runs inference** on each test comment one-by-one:
   - Cleans the text.
   - Tokenizes it with padding/truncation.
   - Passes through the trained model.
   - Converts the predicted integer back to the label string using `inv_map`.

In [ ]:
if __name__ == "__main__":
    
    # ── Step 1: Train ─────────────────────────────────────────────────
    trained_model, tokenizer, device, inv_map = train_model()
    trained_model.eval()  # Switch to evaluation mode (disables dropout)

    # ── Step 2: Load test data ────────────────────────────────────────
    test_df = pd.read_csv("/kaggle/input/competitions/dl-assignment-toxic-comment-classification/test.csv")
    test_ids   = test_df["id"].tolist()
    test_texts = test_df["Toxic Comment"].tolist()

    predictions = []

    # ── Step 3: Inference ─────────────────────────────────────────────
    with torch.no_grad():
        for text in test_texts:
            clean = clean_toxic_text(text)  # Normalize leet characters

            enc = tokenizer(
                clean,
                return_tensors='pt',
                max_length=80,
                truncation=True,
                padding='max_length'
            )

            ids  = enc['input_ids'].to(device)
            mask = enc['attention_mask'].to(device)

            out = trained_model(ids, mask)             # Forward pass
            _, pred = torch.max(out, dim=1)            # Get predicted class index
            predictions.append(inv_map[pred.item()])   # Convert index → label string

    # ── Step 4: Save submission ───────────────────────────────────────
    submission = pd.DataFrame({
        "id": test_ids,
        "Classification": predictions
    })

    submission.to_csv("submission.csv", index=False)
    print("Submission saved as submission.csv")